# Fake News Detector

In [1]:
# Basic imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
# Pre-processing
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
# Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
# Model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aparajaya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Data Loading 
* Load and examine dataset
* Convert datatypes
* Treat missing values

In [2]:
df = pd.read_csv("Dataset.csv")

In [3]:
# Examining Dataset 
print("Dataset shape:",df.shape)
print("\nSample: ")
df.head(1)

Dataset shape: (72134, 4)

Sample: 


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1


In [4]:
# Checking and converting datatypes
print("Original Datatypes:\n",df.dtypes)
df['title'] = df['title'].astype('string')
df['text'] = df['text'].astype('string')
print("\nChanged Datatypes:\n",df.dtypes)

Original Datatypes:
 Unnamed: 0     int64
title         object
text          object
label          int64
dtype: object

Changed Datatypes:
 Unnamed: 0             int64
title         string[python]
text          string[python]
label                  int64
dtype: object


In [5]:
# Treating missing values
print("Number of missing values per column:\n",df.isna().sum())
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')
print("\nAfter treating missing values-")
print("Number of missing values per column:\n",df.isna().sum())

Number of missing values per column:
 Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

After treating missing values-
Number of missing values per column:
 Unnamed: 0    0
title         0
text          0
label         0
dtype: int64


## Preprocessing

In [6]:
def preprocess(df, remove_stopwords=True, remove_sourcewords=False, remove_punctuation=True, lowercase=True, check= False):
    df = df.copy()
    # Convert to Lowercase
    if lowercase:
        df['text'] = df['text'].str.lower()
        df['title'] = df['title'].str.lower()

    # Remove punctuation
    if remove_punctuation: 
        df['title'] = df['title'].str.replace(r'[^\w\s]+', '', regex=True) 
        df['text'] = df['text'].str.replace(r'[^\w\s]+', '', regex=True) 

    # Remove stopwords (Conditional)
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))

        df['text'] = df['text'].apply(
            lambda x: " ".join([word for word in x.split() if word not in stop_words])
        )
        df['title'] = df['title'].apply(
            lambda x: " ".join([word for word in x.split() if word not in stop_words])
        )

    if remove_sourcewords:
        source_tokens = ["reuters", "breitbart", "washington reutors", "new york times"]

        df['text'] = df['text'].apply(
            lambda x: " ".join([word for word in x.split() if word not in source_tokens])
        )
        df['title'] = df['title'].apply(
            lambda x: " ".join([word for word in x.split() if word not in source_tokens])
        )

    # Check if pre-processing caused too many empty rows
    if check: 
        print("No of empty title rows: ",(df['title'].str.len() == 0).sum())
        print("No of empty text rows: ",(df['text'].str.len() == 0).sum())
        count = ((df['title'].str.len() == 0) & (df['text'].str.len() == 0)).sum()
        print("No of rows with empty title AND text: ",count)
    
    return df

## Vectorization 

In [7]:
def vectorize(df,mdf=5, ngramRange=(1,2), max_features_title=50000, max_features_text=500000, vec_text = True, vec_title = True, verbose = False):
    X = df[["title", "text"]]
    Y = df["label"]

    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.33, random_state=42)

    if vec_text and vec_title:
        # Initialize the TF-IDF Vectorizer
        vectorizer_title = TfidfVectorizer(min_df=mdf, ngram_range=ngramRange, max_features=max_features_title)
        vectorizer_text = TfidfVectorizer(min_df=mdf, ngram_range=ngramRange, max_features=max_features_text)
        
        # Fit and transform the data
        tfidf_title_train = vectorizer_title.fit_transform(X_train["title"])
        tfidf_title_test = vectorizer_title.transform(X_test["title"])

        tfidf_text_train = vectorizer_text.fit_transform(X_train["text"])
        tfidf_text_test = vectorizer_text.transform(X_test["text"])

        X_train = hstack([tfidf_title_train, tfidf_text_train])
        X_test = hstack([tfidf_title_test, tfidf_text_test])

        feature_names = np.concatenate((
        vectorizer_title.get_feature_names_out(),
        vectorizer_text.get_feature_names_out()
        ))
    elif vec_title:
        # Initialize the TF-IDF Vectorizer
        vectorizer_title = TfidfVectorizer(min_df=mdf, ngram_range=ngramRange, max_features=max_features_title)
        # Fit and transform the data
        tfidf_title_train = vectorizer_title.fit_transform(X_train["title"])
        tfidf_title_test = vectorizer_title.transform(X_test["title"])
        
        X_train = tfidf_title_train
        X_test = tfidf_title_test

        feature_names = vectorizer_title.get_feature_names_out()
    elif vec_text:
        # Initialize the TF-IDF Vectorizer
        vectorizer_text = TfidfVectorizer(min_df=mdf, ngram_range=ngramRange, max_features=max_features_text)
        # Fit and transform the data
        tfidf_text_train = vectorizer_text.fit_transform(X_train["text"])
        tfidf_text_test = vectorizer_text.transform(X_test["text"])

        X_train = tfidf_text_train
        X_test = tfidf_text_test

        feature_names = vectorizer_text.get_feature_names_out()


    # Matrix inspection
    # Shape = (number_of_documents, number_of_features)
    if verbose:
        print(X_train.shape)
        print(X_test.shape)

    total_features = X_train.shape[1]
    return X_train, X_test, Y_train, Y_test, feature_names, total_features

## Evaluation

In [8]:
def evaluate(Y_test, Y_pred, model, feature_names, plot_cm= False, verbose=True):
    # Metrics
    accuracy = accuracy_score(Y_test, Y_pred)
    precision = precision_score(Y_test, Y_pred)
    recall = recall_score(Y_test, Y_pred)
    f1 = f1_score(Y_test, Y_pred, average="macro")

    # Confusion Matrix
    cm = confusion_matrix(Y_test, Y_pred)

    # Top positive and negative features
    coefficients = model.coef_[0] 

    df_features = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': coefficients
    })

    top_negative_features = df_features.sort_values(by='Coefficient', ascending=True).head(5)
    top_positive_features = df_features.sort_values(by='Coefficient', ascending=False).head(5)

    if verbose:
        print("Accuracy: ", accuracy)
        print("Precision: ", precision)
        print("Recall: ", recall)
        print("F1: ", f1)

        print("\nConfusion Matrix:\n", cm)

        if plot_cm:
            disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Class 0', 'Class 1'])
            disp.plot(cmap=plt.cm.Blues)
            plt.show()

        print("\nTop 5 Negative Features:")
        print(top_negative_features)

        print("\nTop 5 Positive Features:")
        print(top_positive_features)

    return accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features

## Models

### Model 1 (Baseline)
Model - Logistic Regression     

Features -      
* TF-IDF 
    - min_df = 5
    - max_features=50000 for title, 500000 for text
* N-grams
    - Uni + Bigrams
* Text and title processed separately
* Stopwords removed
* Total features = 477k      

Time - 
* Feature Creation time : 44.1s   
* Training Time : 2.5s                

Metrics -           
* Accuracy:  0.96782
* Precision:  0.95957
* Recall:  0.97820
* F1:  0.96779
* Confusion Matrix:         
 [11148     501        -> False Positives            
   265      11891]     -> False Negatives              

In [9]:
# Creating features
start_time = time.perf_counter()

df = preprocess(df, check=True)
X_train, X_test, Y_train, Y_test, feature_names, total_features = vectorize(df)

end_time = time.perf_counter()
creation_time = end_time - start_time

print(f"\nFeatures created in {creation_time:.4f} seconds.")
print("Total features = ",total_features)

No of empty title rows:  563
No of empty text rows:  786
No of rows with empty title AND text:  1

Features created in 42.3756 seconds.
Total features =  477076


In [10]:
# Training model
start_time = time.perf_counter()

model = LogisticRegression(random_state=42).fit(X_train, Y_train)
Y_pred = model.predict(X_test)

end_time = time.perf_counter()
training_time = end_time - start_time

print(f"Training completed in {training_time:.4f} seconds.")

Training completed in 2.5414 seconds.


In [11]:
# Evaluation
evaluate(Y_test, Y_pred, model, feature_names)

Accuracy:  0.9678218861583701
Precision:  0.9595706907682375
Recall:  0.9782000658111221
F1:  0.9677905081677439

Confusion Matrix:
 [[11148   501]
 [  265 11891]]

Top 5 Negative Features:
                   Feature  Coefficient
352973             reuters   -19.644100
361547                said   -18.528369
1810             breitbart   -14.876882
432972              trumps    -9.132123
454520  washington reuters    -7.019568

Top 5 Positive Features:
         Feature  Coefficient
15125      video    10.903366
446457       via     8.507856
1777    breaking     6.366823
205873     image     6.172324
197585   hillary     5.396140


(0.9678218861583701,
 0.9595706907682375,
 0.9782000658111221,
 0.9677905081677439,
 array([[11148,   501],
        [  265, 11891]]),
                    Feature  Coefficient
 352973             reuters   -19.644100
 361547                said   -18.528369
 1810             breitbart   -14.876882
 432972              trumps    -9.132123
 454520  washington reuters    -7.019568,
          Feature  Coefficient
 15125      video    10.903366
 446457       via     8.507856
 1777    breaking     6.366823
 205873     image     6.172324
 197585   hillary     5.396140)

### Model Analysis

#### Feature Importance Observations

* The most influential features for classification were:

  * **Negative coefficients (Real News indicators):** `reuters`, `said`, `washington reuters`, `new york`, `york times`, `president donald`, `twitter`
  * **Positive coefficients (Fake News indicators):** `video`, `via`, `breaking`, `image`, `image via`, `hillary`, `2016`, `october`, `trump`, `november`

#### Interpretation

* The model appears to heavily rely on **source-related terms** such as `reuters`, `washington reuters`, and `breitbart`.
* Several highly weighted features correspond to **publisher names, locations, or news organizations**, suggesting that source identification may contribute significantly to classification performance.
* Features associated with fake news tend to include **media-sharing and clickbait-style terms** such as `video`, `image`, `via`, and `breaking`.
* The large coefficient magnitudes indicate that some features are highly predictive of a particular class.

#### N-gram Analysis

* Although bigrams appeared among the top features (e.g., `washington reuters`, `image via`, `new york`), the majority of the most influential features were still unigrams.
* This suggests that most predictive power may come from individual words, while bigrams provide additional contextual information.
* Further comparison with a unigram-only model is required to quantify the actual contribution of bigrams.

#### Model Behavior

* The model achieved high and balanced performance:

  * Accuracy: 96.78%
  * Precision: 95.96%
  * Recall: 97.82%
  * F1 Score: 96.78%
* Recall is slightly higher than precision, indicating that the model misses relatively few positive samples but produces somewhat more false positives.
* The confusion matrix shows that classification errors are relatively low compared to the total number of samples.

#### Computational Performance

* Training time for Logistic Regression was approximately **2.5 seconds**.
* TF-IDF vectorization (~37 seconds) was significantly more computationally expensive than model training.
* For this pipeline, feature extraction is the primary computational bottleneck rather than classifier training.

## Experiments
* Unigram vs Unigram+Bigram
* Stopwords ON vs OFF
* Feature Engineering
    - Title only
    - Text only
    - Title and Text combined (post and pre-vectorization)
* Varying min df
    - 2
    - 5
    - 10
* Removing obvious news source identifiers

In [12]:
def run_experiment(df, ngramRange=(1,2), remove_stopwords=True, vec_title=True, vec_text=True, mdf=5, remove_sourcewords=False):
    # Creating features
    start_time = time.perf_counter()

    df = preprocess(df,remove_stopwords=remove_stopwords, remove_sourcewords=remove_sourcewords)
    X_train, X_test, Y_train, Y_test, feature_names, total_features = vectorize(df, ngramRange=ngramRange, vec_title=vec_title, vec_text=vec_text, mdf=mdf)

    end_time = time.perf_counter()
    creation_time = end_time - start_time

    print(f"\nFeatures created in {creation_time:.4f} seconds.")
    print("Total features = ",total_features)

    # Training model
    start_time = time.perf_counter()

    model = LogisticRegression(random_state=42).fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    end_time = time.perf_counter()
    training_time = end_time - start_time

    print(f"Training completed in {training_time:.4f} seconds.")
    
    # Evaluation
    accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = evaluate(Y_test, Y_pred, model, feature_names)

    return accuracy, precision, recall, f1, total_features, creation_time, training_time
    

### Experiment 1 (Unigram vs Unigram+Bigram)

In [13]:
accuracy, precision, recall, f1, total_features, creation_time, training_time = run_experiment(df, ngramRange=(1,1))


Features created in 11.9794 seconds.
Total features =  70042
Training completed in 0.6649 seconds.
Accuracy:  0.9671917664356228
Precision:  0.9603399433427762
Recall:  0.976061204343534
F1:  0.9671628837188619

Confusion Matrix:
 [[11159   490]
 [  291 11865]]

Top 5 Negative Features:
         Feature  Coefficient
54379    reuters   -22.127018
55638       said   -16.541791
1202   breitbart   -14.349670
9395        york    -9.069315
63968     trumps    -8.879073

Top 5 Positive Features:
        Feature  Coefficient
9014      video    10.279871
66079       via     9.882855
36281     image     6.699478
1196   breaking     6.195092
46844   october     5.642485


### Experiment 2 (Stopwords ON vs OFF)

In [14]:
accuracy, precision, recall, f1, total_features, creation_time, training_time = run_experiment(df, remove_stopwords=False)


Features created in 38.6401 seconds.
Total features =  477076
Training completed in 2.5536 seconds.
Accuracy:  0.9678218861583701
Precision:  0.9595706907682375
Recall:  0.9782000658111221
F1:  0.9677905081677439

Confusion Matrix:
 [[11148   501]
 [  265 11891]]

Top 5 Negative Features:
                   Feature  Coefficient
352973             reuters   -19.644100
361547                said   -18.528369
1810             breitbart   -14.876882
432972              trumps    -9.132123
454520  washington reuters    -7.019568

Top 5 Positive Features:
         Feature  Coefficient
15125      video    10.903366
446457       via     8.507856
1777    breaking     6.366823
205873     image     6.172324
197585   hillary     5.396140


### Experiment 3 (Title Only)

In [15]:
accuracy, precision, recall, f1, total_features, creation_time, training_time = run_experiment(df, vec_text=False)


Features created in 4.9091 seconds.
Total features =  15985
Training completed in 0.0440 seconds.
Accuracy:  0.8973745011552194
Precision:  0.8860799745607759
Recall:  0.9169134583744653
F1:  0.8972176281700783

Confusion Matrix:
 [[10216  1433]
 [ 1010 11146]]

Top 5 Negative Features:
          Feature  Coefficient
1810    breitbart   -17.018015
15915  york times    -8.657581
11862        says    -7.381339
4836      factbox    -5.977784
15911        york    -5.789854

Top 5 Positive Features:
        Feature  Coefficient
15125     video    15.889239
1777   breaking     7.742029
6293    hillary     6.512331
15407     watch     6.439370
14607    tweets     4.665083


### Experiment 4 (Text Only)

In [16]:
accuracy, precision, recall, f1, total_features, creation_time, training_time = run_experiment(df, vec_title=False)


Features created in 38.8839 seconds.
Total features =  461091
Training completed in 1.0750 seconds.
Accuracy:  0.9554295316110061
Precision:  0.9477762531277747
Recall:  0.9659427443237907
F1:  0.9553864200109012

Confusion Matrix:
 [[11002   647]
 [  414 11742]]

Top 5 Negative Features:
                   Feature  Coefficient
336988             reuters   -24.167961
345562                said   -20.436112
416987              trumps   -10.430314
438535  washington reuters    -8.664267
420072             twitter    -8.057454

Top 5 Positive Features:
          Feature  Coefficient
430472        via    11.603038
181600    hillary     7.371793
189888      image     7.277764
189968  image via     6.267579
269976      obama     5.814341


### Experiment 5 (Text + Title combined before vectorization)

In [17]:
df_combined = pd.DataFrame({
    'title': df['title'],
    'text': df['title'].fillna('') + " " + df['text'].fillna(''),
    'label': df['label'] 
})

accuracy, precision, recall, f1, total_features, creation_time, training_time = run_experiment(df_combined, vec_title=False)


Features created in 39.6940 seconds.
Total features =  471771
Training completed in 0.6580 seconds.
Accuracy:  0.9579920184835119
Precision:  0.9524659312134978
Recall:  0.9659427443237907
F1:  0.9579578135720492

Confusion Matrix:
 [[11063   586]
 [  414 11742]]

Top 5 Negative Features:
                   Feature  Coefficient
344318             reuters   -23.280788
353116                said   -19.821144
55807            breitbart   -11.314811
426380              trumps    -9.295632
448812  washington reuters    -8.894122

Top 5 Positive Features:
          Feature  Coefficient
440262        via    11.479507
440914      video    10.446018
185594    hillary     8.053941
194136      image     7.538882
194219  image via     6.420594


### Experiment 6 (min_df=2)

In [18]:
accuracy, precision, recall, f1, total_features, creation_time, training_time = run_experiment(df, mdf=2)


Features created in 40.7749 seconds.
Total features =  550000
Training completed in 1.0777 seconds.
Accuracy:  0.9675278302877547
Precision:  0.959919191919192
Recall:  0.9772128989799276
F1:  0.9674975993019134

Confusion Matrix:
 [[11153   496]
 [  277 11879]]

Top 5 Negative Features:
           Feature  Coefficient
412950     reuters   -19.861736
422179        said   -18.495374
3627     breitbart   -16.025967
500749      trumps    -9.109845
49681   york times    -7.139926

Top 5 Positive Features:
         Feature  Coefficient
45708      video    11.071569
515119       via     8.534276
3558    breaking     6.268860
255233     image     6.075297
246305   hillary     5.324567


### Experiment 7 (min_df=10)

In [19]:
accuracy, precision, recall, f1, total_features, creation_time, training_time = run_experiment(df, mdf=10)


Features created in 39.3790 seconds.
Total features =  195987
Training completed in 0.8508 seconds.
Accuracy:  0.9684940138626339
Precision:  0.9608112475759535
Recall:  0.9782000658111221
F1:  0.9684646014415952

Confusion Matrix:
 [[11164   485]
 [  265 11891]]

Top 5 Negative Features:
           Feature  Coefficient
143912     reuters   -18.955445
147508        said   -17.695253
899      breitbart   -14.498790
177675      trumps    -9.349118
7832    york times    -6.938440

Top 5 Positive Features:
         Feature  Coefficient
7453       video    10.383834
183475       via     8.543894
886     breaking     6.693966
84237      image     6.145000
80978    hillary     5.431574


### Experiment 8 (Removing obvious source identifiers like reuters,breitbart, washington reuters)

In [21]:

accuracy, precision, recall, f1, total_features, creation_time, training_time = run_experiment(df)


Features created in 41.6336 seconds.
Total features =  477076
Training completed in 2.5799 seconds.
Accuracy:  0.9678218861583701
Precision:  0.9595706907682375
Recall:  0.9782000658111221
F1:  0.9677905081677439

Confusion Matrix:
 [[11148   501]
 [  265 11891]]

Top 5 Negative Features:
                   Feature  Coefficient
352973             reuters   -19.644100
361547                said   -18.528369
1810             breitbart   -14.876882
432972              trumps    -9.132123
454520  washington reuters    -7.019568

Top 5 Positive Features:
         Feature  Coefficient
15125      video    10.903366
446457       via     8.507856
1777    breaking     6.366823
205873     image     6.172324
197585   hillary     5.396140
